# Trading

How the illustrative trading overlay turns a fleet of EVs into a day-ahead position, an intraday
book and an imbalance settlement, and what the resulting "trading P&L" can and can't tell a lead.
Every price and pound here is synthetic or illustrative (`evidence_kind="illustrative_synthetic"`):
none is a tariff, a bid, a real settlement or Axle cash (decision 0005).

**The idea in one paragraph.** Great Britain's own balancing product (P415) settles flexibility in
three stages: a **day-ahead** auction the day before delivery, **intraday** continuous trading up
to an hour before each half-hour, and **imbalance** settlement against a rolling historical
**baseline** once delivery is metered. Axle's trading overlay (`model/market.py`, trading contract
v1 sections 4-5 and 9; decision 0004 items 57-58) follows the same three stages on top of the
kernel's own physics: it never changes what the fleet actually does, it only forecasts, prices and
settles it. Three **strategies** run on the identical physics and the identical random futures —
`perfect_foresight` (an upper bound: it knows the week's outcome already), `da_only` (commits once
and never revisits it) and `full` (day-ahead plus hourly intraday re-positioning) — so any
difference between them is trading skill, not a different fleet.

The notebook calls the same functions the app's full run calls (`market.bl01_lite_baseline`,
`market.positions_kwh`, `forecast.run_forecast`) and reads every default from
`model/assumptions.py`, so a model change shows up here as a changed number or a failed check,
never as stale prose.

**Sections**

1. Three stages, one half-hour
2. The BL01-lite baseline
3. Deviation, true reduction and baseline effect
4. Day-ahead position and the commitment share
5. The ledger: nine buckets, one net
6. Three strategies
7. Trader metrics: capture, value of intraday, firmness, CVaR5, £/MW/yr
8. No free money: the day-ahead/intraday premium
9. No leakage: a later price can't move an earlier decision
10. Sanity checks
11. Limitations

In [1]:
# --- MODEL IMPORTS (one cell) -----------------------------------------------
from dataclasses import replace
from datetime import date

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio

from axle_studio.model import assumptions, market
from axle_studio.model import events as model_events
from axle_studio.model.forecast import run_forecast, run_forecast_from_assumptions
from axle_studio.model.summaries import build_study_slots, build_warmup_slots
from axle_studio.ui.style import MUTED_INK, SERIES_COLOURS, apply_notebook_style

# "notebook_connected" embeds only the figure data and loads plotly.js from a CDN, which keeps the
# committed notebook small. GitHub does not run that script, so every chart has a table under it.
pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 120)
pd.set_option("display.max_rows", 100)
LONDON = "Europe/London"

# The three strategies and the nine ledger buckets read their display names here, not new colours:
# "full" reuses the app's "selected" teal because it is the strategy the rest of the app calls
# smart charging's trading result; "da_only" reuses "normal" (the least active choice); PF reuses
# "difference" (a comparison bound, never a real strategy a trader could run).
STRATEGY_LABELS = {
    "da_only": "Day-ahead only",
    "full": "Full (day-ahead + intraday)",
    "perfect_foresight": "Perfect foresight",
}
STRATEGY_COLOURS = {
    "da_only": SERIES_COLOURS["normal"],
    "full": SERIES_COLOURS["selected"],
    "perfect_foresight": SERIES_COLOURS["difference"],
}
BUCKET_LABELS = {
    "day_ahead_revenue_gbp": "Day-ahead revenue",
    "intraday_pnl_gbp": "Intraday P&L",
    "trading_cost_gbp": "Trading cost (spread)",
    "imbalance_gbp": "Imbalance",
    "baseline_effect_gbp": "Baseline effect",
    "grid_event_payment_gbp": "Grid-event payment",
    "supplier_compensation_gbp": "Supplier compensation",
    "customer_revenue_share_gbp": "Customer revenue share",
    "unmet_charge_penalty_gbp": "Unmet-charge penalty",
    "net_gbp": "Net (illustrative)",
}

## Try it

The fleet runs are small so the notebook stays fast (a few seconds each); raise the counts for
steadier numbers. Two records below aren't in the Edit-assumptions dialog yet — the day-ahead
commitment share and the imbalance premiums — so `run_trading_scenario` (next cell) reaches them
the way `model/assumptions.py` itself documents: `forecast.run_forecast(trading_assumptions=...,
price_assumptions=...)` directly, in place of `run_forecast_from_assumptions(values=...)`.
`PREMIUM_TRY_IT` is section 8's "try it" parameter: the deliberate day-ahead/intraday premium.

In [2]:
START_LOCAL_DATE = date(2026, 1, 12)  # the London date the seven study nights start (a Monday)
VEHICLE_COUNT = 80
WORLD_COUNT = 8  # simulated weeks
PREMIUM_TRY_IT = 5.0  # GBP/MWh: day-ahead deliberately priced below the expected intraday close

DEFAULT_COMMITMENT_SHARE = assumptions.TRADING_OVERLAY["trading.day_ahead_commitment_share"].value
DEFAULT_PREMIUM = assumptions.PRICES["da_id_premium_gbp_per_mwh"].value


def run_trading_scenario(
    start_local_date: date,
    *,
    values: dict[str, object] | None = None,
    trading_overrides: dict[str, float] | None = None,
    price_overrides: dict[str, float] | None = None,
):
    """One action-model run, built the same way ``run_forecast_from_assumptions`` is.

    ``trading_overrides``/``price_overrides`` reach records that are not
    dialog-editable yet (``model/assumptions.py``'s ``TRADING_OVERLAY``
    comment: such a value takes ``forecast.run_forecast(trading_assumptions=
    ..., price_assumptions=...)`` directly, because ``resolve_values`` only
    accepts editable names). Everything else -- settings, the cohort
    fixture, the six-cohort market and trading defaults, the empty events
    table -- is the exact composition the wrapper uses, so a scenario run
    here differs from the app's Run only in the one record under test.
    """

    resolved = assumptions.resolve_values(values)
    settings = assumptions.run_settings(resolved, start_local_date)
    fixture = assumptions.cohort_fixture(resolved)
    inputs = assumptions.forecast_inputs(
        resolved, warmup_days=settings.warmup_days, study_days=settings.study_days
    )
    if trading_overrides:
        inputs["trading_assumptions"] = {**inputs["trading_assumptions"], **trading_overrides}
    if price_overrides:
        inputs["price_assumptions"] = {**inputs["price_assumptions"], **price_overrides}
    events = model_events.event_table((), build_study_slots(start_local_date, settings.study_days))
    result = run_forecast(settings, fixture, **inputs, events=events, model="action")
    return replace(result, assumptions=assumptions.result_assumptions(resolved))


base = run_forecast_from_assumptions(
    START_LOCAL_DATE, values={"vehicle_count": VEHICLE_COUNT, "evaluation_world_count": WORLD_COUNT}
)
setup = pd.DataFrame(
    [
        {
            "policy_id": base.policy_id,
            "evidence_kind": base.evidence_kind,
            "vehicles": base.vehicle_count,
            "simulated weeks": base.world_count,
            "day-ahead commitment share (record)": DEFAULT_COMMITMENT_SHARE,
            "day-ahead/intraday premium (record) £/MWh": DEFAULT_PREMIUM,
            "intraday half spread (record) £/MWh": assumptions.TRADING_OVERLAY[
                "trading.half_spread_gbp_per_mwh"
            ].value,
            "customer revenue share (record)": assumptions.TRADING_OVERLAY[
                "trading.customer_revenue_share"
            ].value,
            "unmet-charge penalty (record) £/kWh": assumptions.TRADING_OVERLAY[
                "trading.unmet_charge_penalty_gbp_per_kwh"
            ].value,
            "baseline history nights (working/non-working, record)": (
                f"{assumptions.TRADING_OVERLAY['trading.baseline_working_nights'].value:g}/"
                f"{assumptions.TRADING_OVERLAY['trading.baseline_non_working_nights'].value:g}"
            ),
        }
    ]
)
setup

,policy_id,evidence_kind,vehicles,simulated weeks,day-ahead commitment share (record),day-ahead/intraday premium (record) £/MWh,intraday half spread (record) £/MWh,customer revenue share (record),unmet-charge penalty (record) £/kWh,"baseline history nights (working/non-working, record)"
0,explicit_synthetic_wholesale_v1,illustrative_synthetic,80,8,0.8,0.0,1.0,0.5,0.79,5/2


## 1. Three stages, one half-hour

GB's own balancing product settles in three stages, and the overlay follows the same shape
(trading contract v1 sections 4.4-4.6; decision 0004 items 57-58):

- **Day-ahead**, at 13:00 London the day before delivery: the trader commits a turn-down position,
  priced at that day's published day-ahead auction price.
- **Intraday**, every whole London hour from publication until an hour before delivery (gate
  closure): the trader re-prices the still-open half-hours against its latest forecast of the
  fleet's own smart plan, and trades the change.
- **Imbalance**, at delivery: whatever the day-ahead-plus-intraday position didn't match against
  what the fleet actually delivered settles at the System Imbalance Price (SIP).

No stage uses information published after its own decision instant (contract section 4.8): the
day-ahead trade only sees prices and events known at 13:00 D-1; each intraday trade only sees the
latest published intraday update; only settlement sees the metered outcome. Section 9 checks that
directly.

The trace below is one real half-hour from the run above: the study slot that sold the most
turn-down at day-ahead. `position_updates` (contract section 5.2) holds every decision that
touched it, from the day-ahead row to the last intraday update before its gate closed.

In [3]:
rep_world = base.representative_world_id
deviation = base.deviation_world_slot
best_slot = (
    deviation.query("world_id == @rep_world")
    .sort_values("position_da_only_kwh", ascending=False)
    .iloc[0]
)
slot_index = int(best_slot["slot_index"])
delivery_london = pd.Timestamp(best_slot["interval_start_london"])

updates = (
    base.position_updates.query("world_id == @rep_world and slot_index == @slot_index")
    .sort_values("decision_utc")
    .reset_index(drop=True)
)

stage_figure = go.Figure()
stage_figure.add_trace(
    go.Scatter(
        x=updates["decision_london"],
        y=updates["position_kwh"].round(2),
        mode="lines+markers",
        line={"shape": "hv", "color": SERIES_COLOURS["selected"], "width": 2.5},
        marker={"size": 7},
        name="Position sold so far (day-ahead, then intraday)",
    )
)
stage_figure.add_trace(
    go.Scatter(
        x=[updates["decision_london"].iloc[0], delivery_london],
        y=[best_slot["settled_kwh"]] * 2,
        mode="lines",
        line={"color": MUTED_INK, "dash": "dash", "width": 1.5},
        name="Settled deviation V (what was actually delivered)",
    )
)
stage_figure.add_trace(
    go.Scatter(
        x=[delivery_london],
        y=[best_slot["settled_kwh"]],
        mode="markers",
        marker={"size": 11, "color": SERIES_COLOURS["difference"], "symbol": "diamond"},
        name="Delivery: the gap tops up at the imbalance price",
    )
)
stage_figure.update_layout(
    title=(
        f"One half-hour's position, {delivery_london:%a %d %b %H:%M} London "
        f"(world {rep_world}, slot {slot_index})"
    ),
    xaxis_title="Decision time (London)",
    yaxis_title="Turn-down position (kWh)",
)
apply_notebook_style(stage_figure, height=380)

In [4]:
stage_table = updates[
    [
        "stage",
        "decision_london",
        "hours_to_gate_closure",
        "position_kwh",
        "trade_kwh",
        "price_gbp_per_mwh",
    ]
].copy()
stage_table["decision_london"] = stage_table["decision_london"].dt.strftime("%a %H:%M")
settlement_row = pd.DataFrame(
    [
        {
            "stage": "imbalance (SIP)",
            "decision_london": delivery_london.strftime("%a %H:%M"),
            "hours_to_gate_closure": 0,
            "position_kwh": best_slot["settled_kwh"],
            "trade_kwh": best_slot["settled_kwh"] - updates["position_kwh"].iloc[-1],
            "price_gbp_per_mwh": best_slot["imbalance_gbp_per_mwh"],
        }
    ]
)
pd.concat([stage_table, settlement_row], ignore_index=True).round(2)

,stage,decision_london,hours_to_gate_closure,position_kwh,trade_kwh,price_gbp_per_mwh
0,day_ahead,Sun 13:00,29,62.54,62.54,121.34
1,intraday,Mon 12:00,6,80.28,17.74,130.31
2,intraday,Mon 13:00,5,80.28,0.00,134.17
3,intraday,Mon 14:00,4,80.28,0.00,140.58
4,intraday,Mon 15:00,3,80.28,0.00,139.96
5,intraday,Mon 16:00,2,80.28,0.00,137.36
6,intraday,Mon 17:00,1,80.42,0.15,134.13
7,intraday,Mon 18:00,0,83.06,2.64,136.66
8,imbalance (SIP),Mon 19:00,0,91.19,8.13,119.34


Read the table alongside the chart: every intraday row before the last one traded 0 kWh whenever
the latest price wasn't worth revising the position for; a non-zero `trade_kwh` row is the trader
buying (or selling back) turn-down at that hour's price. The final `imbalance` row is not a trade
at all — it is the day-ahead-plus-intraday position (`f`) reconciled against what the fleet
actually delivered (`V`) once metered, at the imbalance price. A different slot in a different
world might sell nothing at day-ahead and pick it all up intraday, or sell more than it can
deliver and buy back at a loss: this is one real, but not representative, trace.

## 2. The BL01-lite baseline

Settlement can't ask what a flexed EV *would* have imported — nobody meters a counterfactual — so
GB's P415 product defines a rolling historical baseline instead (BL01: the mean of recent similar
days). `market.bl01_lite_baseline` (contract section 4.1) is that rule, simplified for one fleet:

- night `n`'s class is `working` (Monday-Friday) or `non_working` (Saturday-Sunday), by the
  evening's own date, because the evening carries most of the turn-down;
- each half-hour's baseline is the mean of that half-hour's import over the last
  `trading.baseline_working_nights` (or `..._non_working_nights`) history nights of the same
  class (the setup table above gives the live defaults, 5 and 2);
- only history nights **at or before `n - 2`** count. Night `n - 1` ends at noon on `D_n`, after
  the day-ahead decision at 13:00 on `D_n - 1` — including it would let the day-ahead position
  peek at a night that, at decision time, hasn't happened yet.

A short warm-up can hold no history night of a night's own class (the other class is used instead,
and counted) or no history night at all (baseline 0: nothing is sold or settled that night). The
toy fixture below is the contract's own baseline test (section 5.10 checklist): a constant history
import should come back out as the baseline, exactly.

In [5]:
TOY_START = date(2026, 1, 5)  # a Monday: the toy warm-up spans both working and weekend nights
CONSTANT_IMPORT_KWH = 10.0
toy_warmup_slots = build_warmup_slots(TOY_START, warmup_days=6)
toy_study_slots = build_study_slots(TOY_START, study_days=2)
toy_history_kwh = np.full((1, len(toy_warmup_slots)), CONSTANT_IMPORT_KWH)  # one toy "world"

toy_baseline, toy_fallback_nights, toy_missing_nights = market.bl01_lite_baseline(
    toy_history_kwh,
    toy_warmup_slots,
    toy_study_slots,
    working_nights=int(assumptions.TRADING_OVERLAY["trading.baseline_working_nights"].value),
    non_working_nights=int(
        assumptions.TRADING_OVERLAY["trading.baseline_non_working_nights"].value
    ),
)
pd.DataFrame(
    {
        "local_half_hour": toy_study_slots["local_time_label"],
        "night_index": toy_study_slots["night_index"],
        "day_type": toy_study_slots["day_type"],
        "B0 kWh (should equal the constant import)": toy_baseline[0].round(6),
    }
).iloc[[0, 1, 2, 47, 48, 49]]

,local_half_hour,night_index,day_type,B0 kWh (should equal the constant import)
0,12:00,0,weekday,10.0
1,12:30,0,weekday,10.0
2,13:00,0,weekday,10.0
47,11:30,0,weekday,10.0
48,12:00,1,weekday,10.0
49,12:30,1,weekday,10.0


Now the real run: one representative simulated week's baseline against what the fleet actually
imported on both paths, for the night that settled the most turn-down. `unmanaged_kwh` (`U`, no
Axle action) sits well above the baseline in the evening because BL01's history comes from the
warm-up's *unmanaged* nights (section 4.1: both kernel paths share the warm-up except its last
night); `metered_kwh` (`M`, what the smart fleet actually drew) sits below it where the trader
sold turn-down. The lower panel is the settled volume `V = max(0, B - M)` in the open settlement
window.

In [6]:
best_night = int(
    deviation.query("world_id == @rep_world").groupby("night_index")["settled_kwh"].sum().idxmax()
)
night_rows = deviation.query("world_id == @rep_world and night_index == @best_night").sort_values(
    "slot_index"
)
night_london = night_rows["interval_start_london"].dt.tz_convert(LONDON)

baseline_figure = go.Figure()
for column, label, colour, dash in (
    ("unmanaged_kw", "Unmanaged (U, counterfactual)", SERIES_COLOURS["normal"], "solid"),
    ("baseline_kw", "Settlement baseline (B0 + adjustment)", MUTED_INK, "dash"),
    ("metered_kw", "Metered, this path (M)", SERIES_COLOURS["selected"], "solid"),
):
    baseline_figure.add_trace(
        go.Scatter(
            x=night_london,
            y=night_rows[column].round(2),
            mode="lines",
            line={"color": colour, "dash": dash, "width": 2},
            name=label,
        )
    )
baseline_figure.update_layout(
    title=f"Night {best_night}: baseline against unmanaged and metered fleet import",
    xaxis_title="London local time",
    yaxis_title="kW",
)
apply_notebook_style(baseline_figure, height=360)

In [7]:
settled_figure = go.Figure(
    go.Bar(
        x=night_london,
        y=(night_rows["settled_kwh"] / 0.5).round(2),
        marker_color=SERIES_COLOURS["selected"],
        name="Settled turn-down (V / 0.5 h)",
    )
)
settled_figure.update_layout(
    title=f"Night {best_night}: settled volume V = max(0, B - M)",
    xaxis_title="London local time",
    yaxis_title="kW",
)
apply_notebook_style(settled_figure, height=260)

## 3. Deviation, true reduction and baseline effect

Contract section 4.2 splits every slot's deviation into two pieces that add up exactly:

```
deviation        D = B - M
true reduction   R = U - M            (what smart charging actually moved)
baseline effect  E = B - U            (what the baseline over- or under-states U by)
D = R + E
settled volume   V = max(0, D) inside the settlement window
```

Only `D` is observable at settlement (nobody meters the counterfactual `U`), so `R` and `E` are
reported, not settled, splits of it: `R` is the flexibility a customer actually delivered, `E` is
what the historical baseline happens to say about a night that didn't need to be flexed at all.
The stacked bars below split each settled slot of the same night; a slot can have a negative `E`
even while `V > 0` if the baseline understates that night's own unmanaged import.

In [8]:
settled_night = night_rows.query("settled_kwh > 0")
decomposition_figure = go.Figure()
decomposition_figure.add_trace(
    go.Bar(
        x=settled_night["interval_start_london"].dt.tz_convert(LONDON),
        y=settled_night["true_reduction_kwh"].round(2),
        name="True reduction R = U - M",
        marker_color=SERIES_COLOURS["selected"],
    )
)
decomposition_figure.add_trace(
    go.Bar(
        x=settled_night["interval_start_london"].dt.tz_convert(LONDON),
        y=settled_night["baseline_effect_kwh"].round(2),
        name="Baseline effect E = B - U",
        marker_color=SERIES_COLOURS["difference"],
    )
)
decomposition_figure.update_layout(
    barmode="relative",
    title=f"Night {best_night}: settled slots split into R and E (R + E = D exactly)",
    xaxis_title="London local time",
    yaxis_title="kWh per half-hour",
)
apply_notebook_style(decomposition_figure, height=340)

In [9]:
weekly_split = (
    deviation.query("world_id == @rep_world and settled_kwh > 0")
    .groupby("night_index")[["true_reduction_kwh", "baseline_effect_kwh", "settled_kwh"]]
    .sum()
    .rename(
        columns={
            "true_reduction_kwh": "R: true reduction kWh",
            "baseline_effect_kwh": "E: baseline effect kWh",
            "settled_kwh": "settled kWh (this is D, not R + E, over settled slots only)",
        }
    )
)
weekly_split.round(1)

,R: true reduction kWh,E: baseline effect kWh,"settled kWh (this is D, not R + E, over settled slots only)"
night_index,,,
0,867.0,-169.1,697.9
1,747.4,-23.8,723.6
2,909.2,-128.4,780.8
3,501.2,76.9,578.0
4,479.8,-1.1,478.7
5,592.5,-110.5,482.0
6,243.4,182.9,426.4


## 4. Day-ahead position and the commitment share

The day-ahead position is not the whole forecast deviation: it is a share `c` of it (contract
section 9.2, decision 0004 item 58 (C)), the rest left to intraday (`full`) or to imbalance
(`da_only`):

```
q_t = c x mask_DA,t x max(0, B0_t - x_t)
```

where `x_t` is the fleet's expected smart import from `market.expected_smart_kwh`, applying the
dispatcher's own `action.plan_cheapest_slots` rule to each session's typical window and prices
known at the day-ahead decision. `c = 1` reproduces the plain formula; the default below commits
most, not all, of the forecast, leaving room to true it up intraday rather than lock in a forecast
error many hours before delivery.

Two full runs below differ only in `trading.day_ahead_commitment_share`: everything else,
including the random futures, is identical, so any difference is the commitment share alone.

In [10]:
full_commitment = run_trading_scenario(
    START_LOCAL_DATE,
    values={"vehicle_count": VEHICLE_COUNT, "evaluation_world_count": WORLD_COUNT},
    trading_overrides={"trading.day_ahead_commitment_share": 1.0},
)

commitment_rows = []
for label, result, share in (
    ("default (c = %.2g)" % DEFAULT_COMMITMENT_SHARE, base, DEFAULT_COMMITMENT_SHARE),
    ("c = 1.0 (commit everything at day-ahead)", full_commitment, 1.0),
):
    week = result.trading_week_world.query("strategy == 'full'")
    kpis = result.trading_kpis.query("strategy == 'full'").set_index("metric")
    commitment_rows.append(
        {
            "commitment share": label,
            "sold day-ahead, mean MWh/week": week["sold_day_ahead_mwh"].mean(),
            "final position, mean MWh/week": week["final_position_mwh"].mean(),
            "firmness (delivered / final position)": kpis.at["firmness", "mean"],
            "firmness, day-ahead leg only": kpis.at["firmness_day_ahead", "mean"],
            "imbalance share of traded volume": kpis.at["imbalance_share_of_traded", "mean"],
        }
    )
commitment_table = pd.DataFrame(commitment_rows).round(4)

commitment_figure = go.Figure(
    go.Bar(
        x=commitment_table["commitment share"],
        y=commitment_table["sold day-ahead, mean MWh/week"].round(3),
        marker_color=[SERIES_COLOURS["normal"], SERIES_COLOURS["selected"]],
    )
)
commitment_figure.update_layout(
    title="Day-ahead volume sold scales linearly with the commitment share",
    yaxis_title="MWh sold at day-ahead per week (mean over worlds)",
)
apply_notebook_style(commitment_figure, height=320)

In [11]:
commitment_table

,commitment share,"sold day-ahead, mean MWh/week","final position, mean MWh/week",firmness (delivered / final position),"firmness, day-ahead leg only",imbalance share of traded volume
0,default (c = 0.8),3.4260,3.9737,0.9452,0.8497,0.1496
1,c = 1.0 (commit everything at day-ahead),4.2824,4.0729,0.9452,0.8181,0.1450


Raising `c` to 1 sells more at day-ahead, as expected, but firmness against day-ahead needn't rise
or fall monotonically in a real run the way it does on a controlled, no-forecast-error fixture
(contract section 9.2's test is explicitly stated for that fixture, not for a noisy run): the
table shows what this run's own forecast bias actually does, honestly, rather than assuming the
fixture's clean result carries over.

## 5. The ledger: nine buckets, one net

Every night's trading cash splits into nine buckets that sum exactly to a net figure (decision
0005, contract section 4.7); none of it is "Axle cash". The **baseline effect** is carved out of
day-ahead revenue at the day-ahead price, so it is never mistaken for flexibility actually
delivered, and a **customer revenue share** (the setup table's live default) is paid out of
positive gross P&L before the net is struck. `supplier_compensation_gbp` defaults to £0/MWh
because P415 mutualises it across suppliers rather than charging one aggregator.

The waterfall below uses **means** across simulated weeks, not medians or other quantiles: means
of the buckets add up to the mean net exactly (contract section 5.5); quantiles of nine separately
distributed buckets would not.

In [12]:
ledger_summary = base.trading_ledger_summary
week_full = ledger_summary[
    ledger_summary["night_index"].isna() & (ledger_summary["strategy"] == "full")
].set_index("metric")
bucket_order = list(market.BUCKETS) + ["net_gbp"]
bucket_means = week_full.loc[bucket_order, "mean"]

waterfall = go.Figure(
    go.Waterfall(
        x=[BUCKET_LABELS[name] for name in bucket_order],
        y=bucket_means.round(2),
        measure=["relative"] * len(market.BUCKETS) + ["total"],
        connector={"line": {"color": MUTED_INK}},
        increasing={"marker": {"color": SERIES_COLOURS["selected"]}},
        decreasing={"marker": {"color": SERIES_COLOURS["difference"]}},
        totals={"marker": {"color": MUTED_INK}},
    )
)
waterfall.update_layout(
    title="Illustrative trading P&L, strategy 'full', mean per week across simulated weeks",
    yaxis_title="GBP per week (illustrative)",
)
apply_notebook_style(waterfall, height=420)

In [13]:
week_full.loc[bucket_order, ["unit", "mean", "p10", "p50", "p90"]].rename(
    index=BUCKET_LABELS
).round(2)

,unit,mean,p10,p50,p90
metric,,,,,
Day-ahead revenue,GBP per week,370.69,320.44,372.01,417.98
Intraday P&L,GBP per week,61.90,37.01,68.45,81.89
Trading cost (spread),GBP per week,-2.72,-3.04,-2.80,-2.37
Imbalance,GBP per week,14.29,-5.29,19.65,26.11
Baseline effect,GBP per week,-30.35,-67.07,-31.22,0.49
Grid-event payment,GBP per week,0.00,0.00,0.00,0.00
Supplier compensation,GBP per week,0.00,0.00,0.00,0.00
Customer revenue share,GBP per week,-206.91,-245.09,-202.16,-165.81
Unmet-charge penalty,GBP per week,-6.71,-15.19,-3.60,-2.04


## 6. Three strategies

`perfect_foresight` sells the week's realised settled deviation at day-ahead (an upper bound: real
trading never knows the future); `da_only` commits once at day-ahead and lets everything else
settle at imbalance; `full` adds hourly intraday re-positioning. All three price the same physical
outcome — the kernel never follows the trader (contract section 4.6) — so the spread between them
is entirely what forecasting and re-trading are worth.

In [14]:
net_by_strategy = base.trading_kpis.query("metric == 'net_gbp_per_week'").set_index("strategy")
net_by_strategy = net_by_strategy.loc[list(market.STRATEGIES)]

strategy_figure = go.Figure(
    go.Bar(
        x=[STRATEGY_LABELS[s] for s in net_by_strategy.index],
        y=net_by_strategy["mean"].round(1),
        error_y={
            "type": "data",
            "symmetric": False,
            "array": (net_by_strategy["p90"] - net_by_strategy["mean"]).round(1),
            "arrayminus": (net_by_strategy["mean"] - net_by_strategy["p10"]).round(1),
        },
        marker_color=[STRATEGY_COLOURS[s] for s in net_by_strategy.index],
    )
)
strategy_figure.update_layout(
    title="Illustrative net trading P&L per week, by strategy (mean, P10-P90 whiskers)",
    yaxis_title="GBP per week (illustrative)",
)
apply_notebook_style(strategy_figure, height=340)

In [15]:
base.trading_kpis.query(
    "metric in ['net_gbp_per_week', 'settled_mwh_per_week', 'imbalance_volume_share']"
).pivot(index="metric", columns="strategy", values="mean")[list(market.STRATEGIES)].round(3)

strategy,da_only,full,perfect_foresight
metric,,,
imbalance_volume_share,0.421,0.144,0.000
net_gbp_per_week,201.138,200.203,199.576
settled_mwh_per_week,4.120,4.120,4.120


In [16]:
# How much of the week each strategy actually sells at day-ahead: perfect foresight sells the
# whole realised settled deviation there; full and da_only share the same, smaller, forecast-based
# commitment (section 4), which is what section 8's premium argument leans on.
week_all_strategies = base.trading_week_world.groupby("strategy")["sold_day_ahead_mwh"].mean()
week_all_strategies.loc[list(market.STRATEGIES)].rename("sold at day-ahead, mean MWh/week").rename(
    index=STRATEGY_LABELS
).round(3)

strategy
Day-ahead only                 3.426
Full (day-ahead + intraday)    3.426
Perfect foresight              4.120
Name: sold at day-ahead, mean MWh/week, dtype: float64

## 7. Trader metrics: capture, value of intraday, firmness, CVaR5, £/MW/yr

`trading_kpis` (contract section 9.1) adds a trader's own metrics on top of the ledger. `capture
rate` is deliberately the **ratio of means**, not the mean of per-world ratios (lead decision,
contract section 9.10 Q12): a per-world ratio blows up whenever a world's perfect-foresight margin
is near zero, so the ratio of means is the stable headline and the per-world ratios (kept only
where the denominator is positive) give the P10-P90 spread instead. `value_of_intraday` is a
paired comparison (`full` minus `da_only` in the same world), reported on the `full` row only.
`CVaR5` (the mean of the worst 5% of weeks) is filled for money-per-period rows and left `NaN` for
rates and fractions, where "worst 5%" has no single direction.

**Why capture is close to 1 and value of intraday is slightly negative, even though the
day-ahead/intraday premium defaults to 0 (no free money).** With the premium at 0, intraday and
day-ahead have the same expected price, so there is no average price edge to trading intraday
instead of committing everything at day-ahead — the market is deliberately built so a trader can't
earn a free spread just by holding a book (`da_id_premium_gbp_per_mwh`, section 8). What intraday
trading *does* cost is real: `trading.half_spread_gbp_per_mwh` (the setup table's live default) on
every MWh re-traded. `full` re-trades to correct its day-ahead forecast error, paying that spread
each time, with no offsetting price edge to earn it back — so `value_of_intraday` comes out small
and negative, not because intraday trading is a bad idea, but because in this v1 model precise
delivery earns nothing extra at settlement (a single imbalance price pays over- and
under-delivery alike; contract section 5.10 explicitly does not expect `PF >= full >= DA-only` in
mean net for that reason). Capture stays close to 1 because the day-ahead forecast
(`market.expected_smart_kwh`, itself the dispatcher's own planning rule) is already close to what
the fleet delivers; the small shortfall from 1 is the forecast bias of section 4.3 plus the
intraday spread cost, both modest next to a whole week's flexibility revenue.

In [17]:
trader_metrics = (
    "capture_rate",
    "value_of_intraday_gbp_per_week",
    "cost_of_uncertainty_gbp_per_week",
    "firmness",
    "firmness_day_ahead",
    "imbalance_gbp_per_mwh_traded",
    "flex_margin_gbp_per_mw_year",
    "day_ahead_spread_gbp_per_mwh",
)
trader_table = base.trading_kpis[base.trading_kpis["metric"].isin(trader_metrics)]
trader_pivot = trader_table.pivot(index="metric", columns="strategy", values="mean").loc[
    list(trader_metrics), list(market.STRATEGIES)
]
trader_pivot.rename(columns=STRATEGY_LABELS).round(3)

strategy,Day-ahead only,Full (day-ahead + intraday),Perfect foresight
metric,,,
capture_rate,1.007,1.003,1.000
value_of_intraday_gbp_per_week,NaN,-1.870,NaN
cost_of_uncertainty_gbp_per_week,-3.125,-1.255,0.000
firmness,0.850,0.945,1.000
firmness_day_ahead,0.850,0.850,1.000
imbalance_gbp_per_mwh_traded,22.288,3.851,0.000
flex_margin_gbp_per_mw_year,120701.265,120228.971,119891.315
day_ahead_spread_gbp_per_mwh,95.791,95.791,95.791


In [18]:
fraction_metrics = ("capture_rate", "firmness", "firmness_day_ahead")
fraction_figure = go.Figure()
for strategy in market.STRATEGIES:
    rows = base.trading_kpis.query("strategy == @strategy and metric in @fraction_metrics")
    rows = rows.set_index("metric").loc[list(fraction_metrics)]
    fraction_figure.add_trace(
        go.Bar(
            x=list(fraction_metrics),
            y=rows["mean"].round(3),
            name=STRATEGY_LABELS[strategy],
            marker_color=STRATEGY_COLOURS[strategy],
        )
    )
fraction_figure.update_layout(
    barmode="group",
    title="Capture rate and firmness by strategy (mean over simulated weeks)",
    yaxis_title="fraction",
)
apply_notebook_style(fraction_figure, height=340)

## 8. No free money: the day-ahead/intraday premium

`da_id_premium_gbp_per_mwh` (default 0, the setup table above) shifts the day-ahead price down by
a deliberate, constant amount against the expected intraday close, uniformly across every
half-hour (prices notebook, section 6). A flat shift changes no half-hour's price *ranking*, so
`action.plan_cheapest_slots` picks exactly the same slots and the day-ahead kWh position is
unchanged; only the cash paid for it moves. `da_only` and `full` share the identical day-ahead
position `q` (contract section 9.2), so a premium that raises or lowers day-ahead revenue does so
by the same amount for both — leaving `value_of_intraday` (their paired difference) untouched.
`perfect_foresight` sells more at day-ahead than either (section 6's table), so it absorbs more of
the same shift, which is why `capture_rate` moves a little even though `value_of_intraday` does
not. Try `PREMIUM_TRY_IT` at a few values and re-run this cell.

**This section runs with intraday dispatch off** (`trading.intraday_dispatch` = 0 on both runs,
through `run_trading_scenario`). Dispatch is on by default (decision 0004 item 62(b)): EVs not
locked by the commitment share re-plan hourly on the latest intraday price, which is day-ahead
plus the premium once a slot is published and the expected shape before that (dispatch contract
v1 section 13). So with dispatch on, a premium shifts published and unpublished slots against
each other, moves the dispatched fleet a little and moves `value_of_intraday` with it. Switching
dispatch off isolates the pure pricing argument above.

In [19]:
# Both runs with intraday dispatch off (see the prose above): the premium then changes no
# half-hour's ranking for any EV, so only the cash moves.
DISPATCH_OFF = {"trading.intraday_dispatch": 0.0}
premium_values = {"vehicle_count": VEHICLE_COUNT, "evaluation_world_count": WORLD_COUNT}
premium_base = run_trading_scenario(
    START_LOCAL_DATE, values=premium_values, trading_overrides=DISPATCH_OFF
)
premium_run = run_trading_scenario(
    START_LOCAL_DATE,
    values=premium_values,
    trading_overrides=DISPATCH_OFF,
    price_overrides={"da_id_premium_gbp_per_mwh": PREMIUM_TRY_IT},
)

premium_rows = []
for label, result, premium in (
    (f"default premium ({DEFAULT_PREMIUM:g} GBP/MWh)", premium_base, DEFAULT_PREMIUM),
    (f"try it: {PREMIUM_TRY_IT:g} GBP/MWh", premium_run, PREMIUM_TRY_IT),
):
    kpis = result.trading_kpis.set_index(["strategy", "metric"])
    premium_rows.append(
        {
            "premium": label,
            "day-ahead revenue, full, GBP/week": result.trading_week_world.query(
                "strategy == 'full'"
            )["day_ahead_revenue_gbp"].mean(),
            "day-ahead revenue, perfect foresight, GBP/week": result.trading_week_world.query(
                "strategy == 'perfect_foresight'"
            )["day_ahead_revenue_gbp"].mean(),
            "value of intraday, GBP/week": kpis.at[
                ("full", "value_of_intraday_gbp_per_week"), "mean"
            ],
            "capture rate, full": kpis.at[("full", "capture_rate"), "mean"],
        }
    )
premium_table = pd.DataFrame(premium_rows).round(3)

premium_figure = go.Figure()
premium_figure.add_trace(
    go.Bar(
        x=premium_table["premium"],
        y=premium_table["value of intraday, GBP/week"],
        name="Value of intraday, GBP/week (dispatch off: unchanged by the premium)",
        marker_color=SERIES_COLOURS["selected"],
    )
)
premium_figure.update_layout(
    title="With dispatch off, value of intraday does not move with the premium",
    yaxis_title="GBP per week (illustrative)",
)
apply_notebook_style(premium_figure, height=320)

In [20]:
premium_table

,premium,"day-ahead revenue, full, GBP/week","day-ahead revenue, perfect foresight, GBP/week","value of intraday, GBP/week","capture rate, full"
0,default premium (0 GBP/MWh),371.019,450.211,-1.615,0.997
1,try it: 5 GBP/MWh,352.335,427.622,-1.615,1.006


## 9. No leakage: a later price can't move an earlier decision

The imbalance price (SIP) is the very last price in the chain: it is realised at settlement, after
every day-ahead and intraday decision has already been made. So shifting the imbalance premium
records should move imbalance cash and nothing about the kWh positions those earlier decisions
already committed to. Below, two full runs share every random draw and every day-ahead and
intraday price; only `imbalance_long_premium_gbp_per_mwh` and `..._short_premium_gbp_per_mwh`
differ (large shifts, so a real effect on imbalance cash isn't hiding in rounding).

In [21]:
imbalance_shifted = run_trading_scenario(
    START_LOCAL_DATE,
    values={"vehicle_count": VEHICLE_COUNT, "evaluation_world_count": WORLD_COUNT},
    price_overrides={
        "imbalance_long_premium_gbp_per_mwh": -60.0,
        "imbalance_short_premium_gbp_per_mwh": 80.0,
    },
)

base_week = base.trading_week_world.query("strategy == 'full'").sort_values("world_id")
shifted_week = imbalance_shifted.trading_week_world.query("strategy == 'full'").sort_values(
    "world_id"
)
leakage_checks = {
    "day-ahead position (sold_day_ahead_mwh) is bit-identical": np.array_equal(
        base_week["sold_day_ahead_mwh"].to_numpy(), shifted_week["sold_day_ahead_mwh"].to_numpy()
    ),
    "final position (final_position_mwh) is bit-identical": np.array_equal(
        base_week["final_position_mwh"].to_numpy(), shifted_week["final_position_mwh"].to_numpy()
    ),
    "settled volume (settled_mwh) is bit-identical": np.array_equal(
        base_week["settled_mwh"].to_numpy(), shifted_week["settled_mwh"].to_numpy()
    ),
    "imbalance cash actually moved (the perturbation was real)": not np.isclose(
        base_week["imbalance_gbp"].sum(), shifted_week["imbalance_gbp"].sum()
    ),
}
pd.Series(leakage_checks, name="passed").to_frame()

,passed
day-ahead position (sold_day_ahead_mwh) is bit-identical,True
final position (final_position_mwh) is bit-identical,True
settled volume (settled_mwh) is bit-identical,True
imbalance cash actually moved (the perturbation was real),True


## 10. Sanity checks

Every check below can fail: each one recomputes a contract identity or invariant from this
notebook's own frames, using no number this notebook typed in by hand. `trading_checks` (contract
section 5.5) is the same table the app's P&L lens shows as its own sanity check.

In [22]:
base.trading_checks

,check_id,description,value,tolerance,passed
0,buckets_sum_to_net,Largest |net - sum of the nine buckets| over l...,0.000000e+00,6.038866e-08,True
1,perfect_foresight_zero_imbalance,Largest |imbalance cash| of perfect foresight ...,0.000000e+00,0.000000e+00,True
2,deviation_identity,Largest |D - R - E| over worlds and slots (kWh).,2.486900e-14,1.000000e-09,True
3,mean_intraday_revision,Informative: mean over worlds of the mean full...,-9.588944e-01,9.600491e-01,True
4,p10_ordering,Informative: 1 when P10 weekly net is perfect ...,0.000000e+00,0.000000e+00,True
5,baseline_class_fallback,Informative: study nights whose baseline histo...,0.000000e+00,0.000000e+00,True
6,baseline_missing_nights,Informative: study nights with no baseline his...,0.000000e+00,0.000000e+00,True
7,dispatch_split_sums_to_intraday,Largest |rebalancing + re-optimisation - bucke...,0.000000e+00,2.852765e-08,True


In [23]:
week_all = base.trading_week_world
pf_week = week_all.query("strategy == 'perfect_foresight'")
grid_by_strategy = week_all.pivot(
    index="world_id", columns="strategy", values="grid_event_payment_gbp"
)
penalty_by_strategy = week_all.pivot(
    index="world_id", columns="strategy", values="unmet_charge_penalty_gbp"
)
bucket_sum = week_full.loc[list(market.BUCKETS), "mean"].sum()

checks = {
    "toy BL01 baseline reproduces a constant history exactly": np.allclose(
        toy_baseline, CONSTANT_IMPORT_KWH
    ),
    "toy baseline used no fallback and no missing night": (
        toy_fallback_nights == 0 and toy_missing_nights == []
    ),
    "deviation identity: D = B - M for every world and slot": np.allclose(
        deviation["deviation_kwh"], deviation["baseline_kwh"] - deviation["metered_kwh"]
    ),
    "deviation splits exactly: D = R + E": np.allclose(
        deviation["deviation_kwh"],
        deviation["true_reduction_kwh"] + deviation["baseline_effect_kwh"],
    ),
    "settled volume V >= 0 everywhere": bool((deviation["settled_kwh"] >= -1e-9).all()),
    "V = mask x max(0, D) exactly": np.allclose(
        deviation["settled_kwh"],
        deviation["settlement_open"].to_numpy() * np.maximum(0.0, deviation["deviation_kwh"]),
    ),
    "ledger buckets sum to net (mean, strategy 'full')": np.isclose(
        bucket_sum, week_full.at["net_gbp", "mean"]
    ),
    "perfect foresight has zero imbalance cash, every world": bool(
        np.allclose(pf_week["imbalance_gbp"], 0.0)
    ),
    "perfect foresight has zero trading cost, every world": bool(
        np.allclose(pf_week["trading_cost_gbp"], 0.0)
    ),
    "day-ahead only has zero intraday P&L and trading cost": bool(
        np.allclose(week_all.query("strategy == 'da_only'")["intraday_pnl_gbp"], 0.0)
        and np.allclose(week_all.query("strategy == 'da_only'")["trading_cost_gbp"], 0.0)
    ),
    "grid-event payment is identical across the three strategies": bool(
        grid_by_strategy.nunique(axis=1).eq(1).all()
    ),
    "unmet-charge penalty is identical across the three strategies": bool(
        penalty_by_strategy.nunique(axis=1).eq(1).all()
    ),
    "day-ahead position scales linearly in the commitment share": np.isclose(
        full_commitment.trading_week_world.query("strategy == 'full'")["sold_day_ahead_mwh"].sum(),
        base.trading_week_world.query("strategy == 'full'")["sold_day_ahead_mwh"].sum()
        / DEFAULT_COMMITMENT_SHARE,
        rtol=1e-6,
    ),
    "dispatch off: value of intraday is unchanged by the day-ahead/intraday premium": np.isclose(
        premium_table.loc[0, "value of intraday, GBP/week"],
        premium_table.loc[1, "value of intraday, GBP/week"],
        atol=1e-6,
    ),
    **leakage_checks,
}
failed = [name for name, passed in checks.items() if not passed]
assert not failed, failed
pd.Series(checks, name="passed").to_frame()

,passed
toy BL01 baseline reproduces a constant history exactly,True
toy baseline used no fallback and no missing night,True
deviation identity: D = B - M for every world and slot,True
deviation splits exactly: D = R + E,True
settled volume V >= 0 everywhere,True
"V = mask x max(0, D) exactly",True
"ledger buckets sum to net (mean, strategy 'full')",True
"perfect foresight has zero imbalance cash, every world",True
"perfect foresight has zero trading cost, every world",True
day-ahead only has zero intraday P&L and trading cost,True


## 11. Limitations

- Prices are synthetic and national (prices notebook): every world's day-ahead forecast ranks the
  same national curve, so the day-ahead position has no local or site constraint to trade against.
- The day-ahead forecast uses deterministic typical clocks (section 4.3), which peaks the expected
  arrival slots more sharply than the fleet actually delivers; imbalance and intraday re-trading
  show that bias rather than hide it (it is part of why capture isn't exactly 1).
- A single imbalance price settles both over- and under-delivery, so this v1 model does not expect
  `perfect_foresight >= full >= da_only` in mean net (contract section 5.10); precise delivery
  earns no extra settlement reward on its own here.
- The baseline-trap switch and the hold-out control group (contract section 9.5) model BL01's
  real decay under repeated flexing; both are out of this notebook's scope, so the baseline
  here always uses the (unflexed) warm-up history. The code names the switch
  `trading.baseline_trap`; the contract calls the same thing `trading.baseline_history`.
- Non-response, grid events and zones (contract sections 2-3, 9.5-9.6) are switched off in every
  run above; each has its own concept notebook or a later wave.
- The small runs here (80 EVs, 8 simulated weeks) are for speed: their P10-P90 spreads are wider
  and `capture_rate`'s per-world quantiles rougher than the app's default run.